In [ ]:
# Cell 1 (only one needed now)
from src import (
    extract_logon_features,
    extract_http_features,
    extract_email_features,
    fuse_feature_matrices,
    InsiderThreatDetector,
)

In [ ]:
# Cell 2 — extraction
hist_logon = extract_logon_features("data/r4.2/logon.csv", "2010-01-01", "2010-07-01", dev_mode=True)
hist_http  = extract_http_features("data/r4.2/http.csv",   "2010-01-01", "2010-07-01", dev_mode=True)
hist_email = extract_email_features("data/r4.2/email.csv", "2010-01-01", "2010-07-01", dev_mode=True)

live_logon = extract_logon_features("data/r4.2/logon.csv", "2011-04-01", None, dev_mode=True)
live_http  = extract_http_features("data/r4.2/http.csv",   "2011-04-01", None, dev_mode=True)
live_email = extract_email_features("data/r4.2/email.csv", "2011-04-01", None, dev_mode=True)

In [ ]:
import pandas as pd
df = pd.read_parquet(hist_logon)
print(df.columns.tolist())  # should include 'total_event_buckets' — the H5 rename
print(df.index.names)        # should be ['user', 'activity_date'

In [ ]:
# Cell 3 — fusion
fused_hist = fuse_feature_matrices(hist_logon, hist_http, hist_email)
fused_live = fuse_feature_matrices(live_logon, live_http, live_email)

In [ ]:
df = pd.read_parquet(fused_hist)
print(df.shape)
print(df.columns.tolist())  # should be all 15 features in MASTER_FEATURE_SCHEMA

In [ ]:
# Cell 4 — training
detector = InsiderThreatDetector(mad_threshold=3.5, missing_data_tolerance=0.2)
pkl_path = detector.fit_baseline(fused_hist)
print(f"Trained model saved to: {pkl_path}")

In [ ]:
import joblib
d = joblib.load(pkl_path)
print("is_fitted:           ", d.is_fitted)
print("features:            ", list(d.baseline_stats.keys()))
print("iso_threshold:       ", d.iso_threshold)
print("has train_percentiles:", hasattr(d, "train_score_percentiles"))
print("model_version:       ", d.model_version)

In [ ]:
import pandas as pd

df = pd.read_parquet("features/scored_features_latest.parquet")

print(f"Rows scored: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print()
print("Threat summary:")
print(f"  confirmed_threat : {df['confirmed_threat'].sum()}")
print(f"  high_risk_review : {df['high_risk_review'].sum()}")
print(f"  data_loss_ioc    : {df['data_loss_ioc'].sum()}")
print()
print("risk_score distribution (calibrated percentiles — C2):")
print(df['risk_score'].describe())
print()
print("Top 5 users by risk_score:")
print(df.nlargest(5, 'risk_score')[['user', 'activity_date', 'risk_score',
                                     'confirmed_threat', 'high_risk_review']])

In [ ]:
import pandas as pd

# 1. Confirm what fraction of training scores fall below the threshold
import joblib
det = joblib.load("iso_pipeline_v20260515_211112.pkl")
percentiles = det.train_score_percentiles
threshold = det.iso_threshold
below = sum(1 for p in percentiles if p < threshold) / len(percentiles)
print(f"Fraction of training distribution below ISO threshold: {below:.1%}")

# 2. Check the MAD-zero feature distributions in training data
import pandas as pd
df = pd.read_parquet("features/master_features_unscored_20260515_211015.parquet")
for col in ['total_event_buckets', 'off_hours_ratio', 'keyword_match_indicator',
            'after_hours_browsing', 'large_attachment_count']:
    print(f"{col}:")
    print(f"  median={df[col].median()}, zero_rate={(df[col]==0).mean():.1%}")

# 3. How many users flagged on a typical day
scored = pd.read_parquet("features/scored_features_latest.parquet")
print(f"\nMAD flag distribution:")
print(scored['mad_score_count'].describe())
print(f"\nISO flag rate: {scored['iso_forest_flag'].mean():.1%}")
print(f"MAD critical rate: {scored['mad_critical_flag'].mean():.1%}")